# DataPilot AI — Dataset A collection (Colab-ready)

**What:** Inspect the curated official-doc inventory and the raw HTML already stored as Dataset A.

**Why:** RAG must be grounded in attributable BI/Data Engineering documentation. The project **does not** scrape entire websites. Only URLs listed in `config/sources.yaml` are collected, with `robots.txt`, delay, and provenance.

**What the results mean:** **42/42** curated pages are on disk under `data/raw/`. Re-running the collector **skips** files that already exist (unless `--force`). This notebook is for academic traceability, not a second crawl.

GPU is **not** required. Next notebook: `02_data_preprocessing.ipynb`.

## 1. Setup

On Colab: mount Drive and point at the synced `Masters_Project` folder. Locally: run from the repo root.

In [ ]:
from pathlib import Path
import json
import os
import sys
from collections import Counter

PROJECT_DIR = "/content/drive/MyDrive/Masters_Project"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT_DIR)
except ImportError:
    pass

ROOT = Path.cwd()
if not (ROOT / "config" / "sources.yaml").exists() and (ROOT.parent / "config" / "sources.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT.resolve())

In [ ]:
# Needed for !python cells on Colab (os.chdir does not apply to the shell)
import os
from pathlib import Path
print("cwd:", Path.cwd())

## 2. Curated inventory

**What:** Count topic URLs per official source from `config/sources.yaml` (version `0.1.2`).

**Why:** The research design is a **knowledge hub**, not a web crawl. Each URL is a chosen SQL / warehouse / BI / orchestration / dbt page.

**Meaning:** Six sources, **42** topic URLs. Expanding the list is allowed only if evaluation shows a coverage gap.

In [ ]:
from src.utils.config import load_sources_config
from src.ingestion.inventory import list_topic_targets
import pandas as pd

cfg = load_sources_config()
targets = list_topic_targets(cfg)
print("inventory_version:", cfg.get("inventory_version"))
print("respect_robots:", cfg.get("politeness", {}).get("respect_robots_txt"))
print("delay_seconds:", cfg.get("politeness", {}).get("default_delay_seconds"))
print("n_targets:", len(targets))

rows = [{
    "source_id": t.source_id,
    "source_name": t.source_name,
    "category": t.category,
    "topic": t.topic,
    "url": t.url,
} for t in targets]
df = pd.DataFrame(rows)
display(df.groupby(["source_id", "category"]).size().rename("n_urls").reset_index())
display(df.head(8))

## 3. Collection policy (do not skip in the write-up)

| Rule | How it is implemented |
|------|------------------------|
| Official pages only | URLs hard-listed in `config/sources.yaml` |
| No full-site scrape | Collector never follows unlisted links |
| robots.txt | `src/ingestion/robots.py` unless `--ignore-robots` |
| Rate limit | default **1.5 s** delay between requests |
| Provenance | `data/raw/meta/<source>/<topic>.json` + HTML path |
| Restartable | existing HTML+meta → status `skipped` / `already_collected` |

CLI: `python scripts/collect_documents.py` (`--dry-run`, `--limit`, `--source postgresql`).

## 4. Dry-run (no network)

**What:** List the 42 targets the collector would fetch.

**Why:** Confirms the notebook is attached to the real project without hitting documentation sites.

**Meaning:** `total_targets` should be **42**. A dry-run does not change `data/raw/`.

In [ ]:
from src.ingestion.collector import DocumentCollector

summary = DocumentCollector().collect(dry_run=True)
printable = {k: v for k, v in summary.items() if k != "records"}
print(json.dumps(printable, indent=2))

## 5. What is already on disk

**What:** Count HTML files and sample one metadata record.

**Why:** Collection already ran successfully (42 HTML files). The academic artefact is the stored corpus plus provenance, not a live re-download in the demo.

**Meaning:** 42 HTML files = inventory fully collected. `collection_summary.json` may show `skipped: 42` on a later run because files already exist — that is success, not a failed crawl.

Do **not** run `--force` unless you intend to re-hit every official site.

In [ ]:
html_files = list((ROOT / "data" / "raw" / "html").rglob("*.html"))
meta_files = list((ROOT / "data" / "raw" / "meta").rglob("*.json"))
print("html files:", len(html_files))
print("meta files:", len(meta_files))
print("by source:", dict(Counter(p.parent.name for p in html_files)))

sample_meta = ROOT / "data" / "raw" / "meta" / "postgresql" / "pg_select.json"
meta = json.loads(sample_meta.read_text(encoding="utf-8"))
keep = ["document_id", "source_name", "title", "url", "category", "collected_at", "html_path"]
print("sample provenance:")
print(json.dumps({k: meta.get(k) for k in keep}, indent=2))

## 6. Optional: collect only if something is missing

Uncomment if HTML count is not 42. Prefer `--limit 1` first. Full collection respects robots and ~1.5 s delays (~one minute for 42 pages).

In [ ]:
# Uncomment only if html files != 42
# !python scripts/collect_documents.py --limit 1 -v

## 7. Takeaways for the dissertation

- Dataset A is a **curated** BI Knowledge Hub (PostgreSQL, Redshift, Power BI, Superset, Airflow, dbt).
- Provenance (URL, source, topic, timestamp) is stored per document so RAG citations are auditable.
- Duplicate official pages (e.g. several Superset topics pointing at one guide) are handled in **preprocessing**, not by inventing extra pages.

**Next:** `notebooks/02_data_preprocessing.ipynb` (clean → 37 accepted docs → 281 chunks → FAISS).